# Brain Tumor MRI Classification — Research Enhancement V2

This notebook is a research-focused improvement of the original project. It preserves the four-class EfficientNetB0 approach while improving reproducibility, validation integrity, training control, and evaluation.

**Important:** Run this notebook in the environment where the dataset is available. The dataset path below matches the original Kaggle notebook. Confirm that the test directory is independent and check for patient-level overlap before making clinical or patient-independent claims.

## 1. Reproducibility and imports

In [ ]:
import os, random, json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger
from sklearn.metrics import (
    classification_report, confusion_matrix, balanced_accuracy_score,
    roc_auc_score, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print('Deterministic TensorFlow operations enabled where supported.')
except Exception as exc:
    print('Deterministic operations could not be enabled:', exc)

print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)

## 2. Dataset paths and experiment configuration

The paths below are retained from the original Kaggle notebook. Change them only if your dataset is stored elsewhere.

In [ ]:
TRAIN_DIR = Path('/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training')
TEST_DIR = Path('/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.15
INITIAL_EPOCHS = 20
FINE_TUNE_EPOCHS = 15
NUM_CLASSES = 4
OUTPUT_DIR = Path('results/research_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for folder in (TRAIN_DIR, TEST_DIR):
    if not folder.exists():
        raise FileNotFoundError(f'Dataset directory not found: {folder}')

print('Training directory:', TRAIN_DIR)
print('Testing directory:', TEST_DIR)
print('Artifacts will be saved to:', OUTPUT_DIR.resolve())

## 3. Corrected data pipelines

**Key improvement:** training images receive augmentation, but validation and test images do not. The training and validation generators use the same split seed and validation fraction so their subsets remain consistent. The test generator is not augmented and does not shuffle, preserving label/prediction order.

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    validation_split=VALIDATION_SPLIT
)

# Separate generator: preprocessing only, no random augmentation.
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VALIDATION_SPLIT
)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    str(TRAIN_DIR), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=SEED
)
val_gen = val_datagen.flow_from_directory(
    str(TRAIN_DIR), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=SEED
)
test_gen = test_datagen.flow_from_directory(
    str(TEST_DIR), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

class_names = list(train_gen.class_indices.keys())
assert train_gen.class_indices == val_gen.class_indices == test_gen.class_indices, 'Class mappings differ across data splits.'
assert len(class_names) == NUM_CLASSES, f'Expected {NUM_CLASSES} classes, found {len(class_names)}.'

print('Class mapping:', train_gen.class_indices)
print('Training images:', train_gen.samples)
print('Validation images:', val_gen.samples)
print('Test images:', test_gen.samples)

## 4. Inspect class distribution

Record the actual image counts produced by the directory iterators. Use these observed counts in the report rather than hard-coding counts from an older README.

In [ ]:
def class_counts(generator, names):
    counts = np.bincount(generator.classes, minlength=len(names))
    return pd.DataFrame({'class': names, 'images': counts})

count_table = class_counts(train_gen, class_names).merge(
    class_counts(val_gen, class_names), on='class', suffixes=('_train', '_validation')
).merge(class_counts(test_gen, class_names), on='class')
count_table = count_table.rename(columns={'images': 'images_test'})
display(count_table)
count_table.to_csv(OUTPUT_DIR / 'dataset_class_counts.csv', index=False)

## 5. Build the EfficientNetB0 transfer-learning model

The original backbone and four-class classification head are retained to make the research comparison fair. The backbone is initially frozen.

In [ ]:
base_model = EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.20)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = models.Model(inputs, outputs, name='efficientnetb0_brain_tumor_v2')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 6. Controlled training with callbacks

Early stopping reduces unnecessary training when validation loss stops improving. The best validation-loss checkpoint is saved separately from the final model. Training logs are saved for reporting.

In [ ]:
best_model_path = OUTPUT_DIR / 'best_model.keras'
callbacks = [
    ModelCheckpoint(str(best_model_path), monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1),
    CSVLogger(str(OUTPUT_DIR / 'training_log.csv'), append=False)
]

history_initial = model.fit(
    train_gen, validation_data=val_gen, epochs=INITIAL_EPOCHS,
    callbacks=callbacks
)

## 7. Fine-tuning

Unfreeze only the final 20 backbone layers and keep Batch Normalization layers frozen. A lower learning rate is used to reduce the risk of damaging pretrained features.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fine_callbacks = [
    ModelCheckpoint(str(best_model_path), monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1),
    CSVLogger(str(OUTPUT_DIR / 'fine_tuning_log.csv'), append=False)
]
history_fine = model.fit(
    train_gen, validation_data=val_gen, epochs=FINE_TUNE_EPOCHS,
    callbacks=fine_callbacks
)

model.save(OUTPUT_DIR / 'final_model.keras')

## 8. Plot training curves

In [ ]:
def plot_history(histories, metric, title):
    plt.figure(figsize=(8, 5))
    for label, hist in histories:
        if metric in hist.history:
            plt.plot(hist.history[metric], label=f'{label} train')
        val_metric = f'val_{metric}'
        if val_metric in hist.history:
            plt.plot(hist.history[val_metric], linestyle='--', label=f'{label} validation')
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

histories = [('Initial', history_initial), ('Fine-tuning', history_fine)]
plot_history(histories, 'accuracy', 'Training and validation accuracy')
plot_history(histories, 'loss', 'Training and validation loss')

## 9. Held-out test evaluation

The test set is evaluated after training. Report per-class precision, recall, and F1-score, as well as overall accuracy and balanced accuracy. These results must be generated by running this notebook; no performance values are assumed here.

In [ ]:
test_gen.reset()
y_true = test_gen.classes
probabilities = model.predict(test_gen, verbose=1)
y_pred = np.argmax(probabilities, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))
print('Test accuracy:', float(np.mean(y_true == y_pred)))
print('Balanced accuracy:', balanced_accuracy_score(y_true, y_pred))

report = classification_report(
    y_true, y_pred, target_names=class_names, digits=4,
    zero_division=0, output_dict=True
)
pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / 'classification_report.csv')

try:
    y_true_binary = label_binarize(y_true, classes=np.arange(len(class_names)))
    macro_auc = roc_auc_score(y_true_binary, probabilities, average='macro', multi_class='ovr')
    print('Macro one-vs-rest ROC-AUC:', macro_auc)
except ValueError as exc:
    print('ROC-AUC could not be calculated:', exc)

## 10. Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Research V2 — Test Confusion Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=200)
plt.show()

## 11. Save experiment metadata

This file records the configuration and class mapping alongside the outputs. It does not replace documenting the dataset source, licensing, split method, and any patient-level grouping information.

In [ ]:
metadata = {
    'project': 'Brain Tumor MRI Classification — Research Enhancement V2',
    'seed': SEED,
    'image_size': list(IMG_SIZE),
    'batch_size': BATCH_SIZE,
    'validation_split': VALIDATION_SPLIT,
    'initial_epochs_requested': INITIAL_EPOCHS,
    'fine_tune_epochs_requested': FINE_TUNE_EPOCHS,
    'classes': class_names,
    'class_indices': train_gen.class_indices,
    'train_images': int(train_gen.samples),
    'validation_images': int(val_gen.samples),
    'test_images': int(test_gen.samples),
    'tensorflow_version': tf.__version__,
    'seed_note': 'Single fixed-seed run; repeat with multiple seeds for stronger research evidence.',
    'data_leakage_note': 'Patient-level independence and duplicate-image overlap must be checked separately if patient identifiers or image hashes are available.'
}
with open(OUTPUT_DIR / 'experiment_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Research V2 artifacts saved in:', OUTPUT_DIR.resolve())

## 12. Research limitations and reporting checklist

Before presenting results:

1. Run the notebook from top to bottom in the target environment.
2. Confirm that the train, validation, and test folders contain the intended images and matching class names.
3. Verify that no patient or near-duplicate image appears across training and test sets, where identifiers or image hashes permit this check.
4. Repeat training with multiple random seeds and report mean and standard deviation for key metrics.
5. Inspect false-negative cases, especially for each tumor class. MRI image classification is an academic decision-support experiment, not a clinical diagnostic tool.
6. Compare this V2 model against the original baseline using the same held-out test set and clearly documented protocol.

**Do not claim improved accuracy until the notebook has been executed and the baseline and V2 results have been compared under the same test protocol.**